# J-lens vs logit lens on GPT-2 XL — shared-activation dataset evaluation

Based on `notebooks/logit_lens/logit_lens_dataset.ipynb`, retaining every metric and diagnostic
across all six evaluation sets. For each prompt, **one forward pass** records the pre-`ln_f`
output of every block. The logit lens reads `unembed(h_l)`; the J-lens reads
`unembed(J_l @ h_l)`. Both use the same activations, tokenization, readout position, controls,
and exact final model logits. The final row is identical; compare inner layers separately.

Load a fitted lens for the selected model, or fit one on an independent mixed corpus below.
Results retain a `lens` column throughout; model accuracy is counted once per item.

**Protocol** (`data/evaluations/README.md`):

* **readout position** — one token per prompt: the last prompt token (the one right before `target`,
  the closing period, the tail of the typo); for poetry — the newline at the end of the first line;
* **rank** — 1-based rank of a word among all 50 257 tokens at that position, min over its single-token
  spellings (with/without leading space, case); for order-ops — min over the synonym set
  (numbers → digits and words, operations → symbols and words); multi-token words fall back to their first token;
* **pass@k** — mean over items of the fraction of `intermediates` whose min-over-layers rank ≤ k;
* **control** — the same ranks for the intermediates of *another* item of the same dataset.
  It is the chance level: a lens that surfaces any vaguely topical word scores here too.

## Setup

In [ ]:
import os
import sys
from pathlib import Path

REPO_URL = "https://github.com/ai360-project-school-j-lens/jacobian-lens-gpt2.git"
REPO_DIR = os.path.abspath("jacobian-lens-gpt2")

# ноутбук открыт из клона репозитория (notebooks/<lens>/) — ищем его корень выше по дереву
local_repo = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "gpt2" / "lens_eval.py").is_file()), None)
if local_repo is not None:
    REPO_DIR = str(local_repo)
elif os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone {REPO_URL} {REPO_DIR}
sys.path.append(REPO_DIR)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

import jlens
from gpt2 import GPT2LensModel
from gpt2.lens_eval import (
    DATASETS,
    FIT_MIX,
    ROLE_NAMES,
    evaluate_paired,
    first_layer,
    fit_with_progress,
    identity_lens,
    layer_hit_rate,
    layer_mean,
    layer_median_rank,
    load_eval,
    load_fit_prompts,
    pass_at_k,
    plot_layer_curves,
    shown_layers,
)

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 240)

In [ ]:
MODEL_ID = "openai-community/gpt2-xl"  # 1.5B parameters, 48 layers, d_model 1600
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

gpt = GPT2LensModel.from_pretrained(MODEL_ID, device=DEVICE)
LENS_NAMES = ("logit lens", "J-lens")
LAYER_STRIDE = 4
assert gpt.n_layers >= 2, "The inner-layer comparison needs at least two layers."
gpt

In [ ]:
evals = {dataset: load_eval(REPO_DIR, dataset) for dataset in DATASETS}

K = 10
KS = sorted({1, 5, 10, 100, K})
LAST = gpt.n_layers - 1
{dataset: len(samples) for dataset, samples in evals.items()}

## Load or fit the J-lens

Set `LENS_PATH` to an existing `JacobianLens.save` file fitted for **this model and all its
inner layers** (e.g. the checkpoint saved by `jacobian_lens.ipynb`). A fit-resume checkpoint
is a different format. Width and layer coverage are checked, but a saved lens does not
record its model ID, so choosing the matching model is necessary.

When the file is absent, stream a mixed corpus using `FIT_MIX`, fit the Jacobians, and save
them. Evaluation prompts are not used for fitting. Loading an existing file skips corpus
downloads. GPT-2 XL fitting is expensive: tune `FIT_FRACTION`, `DIM_BATCH`, and `MAX_SEQ_LEN`
for your hardware, or use a smaller `MODEL_ID`. Resuming requires the same prompt list.

In [ ]:
LENS_PATH = Path(REPO_DIR) / f"{MODEL_ID.split('/')[-1]}_jacobian_lens_dataset.pt"
FIT_FRACTION = 0.25
DIM_BATCH = 8
MAX_SEQ_LEN = 128

if LENS_PATH.is_file():
    fitted_lens = jlens.JacobianLens.load(str(LENS_PATH))
else:
    fit_mix = {source: max(1, round(n * FIT_FRACTION)) for source, n in FIT_MIX.items()}
    fit_prompts = load_fit_prompts(fit_mix, seed=0)
    display(fit_prompts.source.value_counts().to_frame("prompts"))
    fitted_lens = fit_with_progress(
        gpt,
        fit_prompts.text.tolist(),
        dim_batch=DIM_BATCH,
        max_seq_len=MAX_SEQ_LEN,
        checkpoint_path=str(LENS_PATH.with_name(LENS_PATH.stem + "_ckpt.pt")),
        checkpoint_every=5,
    )
    fitted_lens.save(str(LENS_PATH))

assert fitted_lens.d_model == gpt.d_model
assert set(range(LAST)).issubset(fitted_lens.source_layers)
fitted_lens

### Identity sanity check

With `J_l = I`, both readouts must produce the same ranks, best layers, top-1 tokens,
agreement, copy rate, and KL. This small check uses the same paired evaluator as the full run.

In [ ]:
sanity_evals = {"order-ops": evals["order-ops"][:2]}
sanity_words, sanity_items = evaluate_paired(gpt, identity_lens(gpt), sanity_evals, desc="identity check")
for frame in (sanity_words, sanity_items):
    left = frame[frame.lens == "logit lens"].drop(columns="lens").reset_index(drop=True)
    right = frame[frame.lens == "J-lens"].drop(columns="lens").reset_index(drop=True)
    pd.testing.assert_frame_equal(left, right)
del sanity_words, sanity_items
print("Identity lens: all metrics match.")

## 1. Run

`words` — a row per (lens, item, word) with the rank at every layer; `items` — a row per
(lens, item) with the model answer, readout top-1 tokens, and per-layer statistics over all
prompt positions. `evaluate_paired` records activations once per prompt and reads each lens
sequentially, releasing vocabulary logits after collecting its metrics. It does not cache
activations or logits across the dataset.

In [ ]:
words, items = evaluate_paired(gpt, fitted_lens, evals)
words["inner_best"] = [int(r[:-1].min()) for r in words.ranks]
words["inner_best_layer"] = [int(r[:-1].argmin()) for r in words.ranks]

# The final row and model answers must agree for every paired item/word.
word_keys = ["dataset", "item", "kind", "word", "role"]
final_ranks = words.assign(final_rank=[int(r[-1]) for r in words.ranks]).pivot(
    index=word_keys, columns="lens", values="final_rank"
)
assert (final_ranks["logit lens"] == final_ranks["J-lens"]).all()
for column in ("model_top1", "model_correct", "readout_token"):
    paired = items.pivot(index=["dataset", "item"], columns="lens", values=column)
    pd.testing.assert_series_equal(paired["logit lens"], paired["J-lens"], check_names=False)
for name in LENS_NAMES:
    frame = items[items.lens == name]
    np.testing.assert_allclose(np.stack(frame.agreement)[:, -1], 1)
    np.testing.assert_allclose(np.stack(frame.kl_to_final)[:, -1], 0, atol=1e-6)
print("Final model readout matches for both lenses.")

In [ ]:
words.head()

In [ ]:
items.head()

In [ ]:
inter = words[words.kind == "intermediate"]
control = words[words.kind == "control"]
target = words[words.kind == "target"]

## 2. Can GPT-2 XL solve the tasks at all?

The lens can only show an intermediate step the model actually computes. For the three sets with a `target`,
check whether the model's own top-1 at the readout position is the target.

In [ ]:
model_items = items.drop_duplicates(["dataset", "item"])
with_target = model_items.dropna(subset=["model_correct"]).astype({"model_correct": bool})
accuracy = with_target.groupby("dataset").model_correct.mean()
accuracy.to_frame("model top-1 == target")

In [ ]:
with_target.groupby("dataset").head(6)[["dataset", "item", "prompt", "target", "model_top1", "model_correct"]]

## 3. pass@k: intermediates vs control vs target

In [ ]:
scores = pass_at_k(words, ks=KS)
scores.pivot_table(index="dataset", columns=["kind", "k", "lens"], values="score").round(3)

The min over layers includes the last row — the model's own output. Split it: what do the inner layers (all but the last)
show that the output does not?

In [ ]:
inner = pass_at_k(words, ks=KS, layers=slice(None, -1)).assign(readout=f"layers 0..{LAST - 1}")
output = pass_at_k(words, ks=KS, layers=[LAST]).assign(readout="model output")
split = pd.concat([inner, output], ignore_index=True)
split[split.k == K].pivot_table(index="dataset", columns=["kind", "readout", "lens"], values="score").round(3)

In [ ]:
at_k = scores[scores.k == K].pivot(index="dataset", columns=["kind", "lens"], values="score")
at_k.plot.bar(figsize=(13, 4), rot=0, title=f"Both lenses: pass@{K} (min over all layers)")
plt.ylabel(f"pass@{K}")
plt.show()

inner_at_k = inner[inner.k == K].pivot(index="dataset", columns=["kind", "lens"], values="score")
output_at_k = output[output.k == K].pivot(index="dataset", columns=["kind", "lens"], values="score")
inner_delta = pd.DataFrame({
    kind: inner_at_k[(kind, "J-lens")] - inner_at_k[(kind, "logit lens")]
    for kind in ("intermediate", "control", "target")
})
display(inner_delta.rename_axis(columns="J-lens − logit lens, inner pass@k").round(3))

## 4. At what depth do the words surface?

In [ ]:
def word_curves(stat):
    return {
        f"{name}: {kind}": stat(words[(words.lens == name) & (words.kind == kind)])
        for name in LENS_NAMES
        for kind in ("intermediate", "control", "target")
    }


plot_layer_curves(
    word_curves(lambda frame: layer_hit_rate(frame, K)),
    title=f"Both lenses: share of words with rank ≤ {K} at the readout position",
    ylabel=f"rank ≤ {K}",
)
plot_layer_curves(
    word_curves(layer_median_rank),
    title="Both lenses: median rank at the readout position",
    ylabel="median rank",
    logy=True,
)

Where the best rank of each intermediate is reached (only words that are hits, rank ≤ K):

In [ ]:
(
    inter[inter.best_rank <= K]
    .groupby(["dataset", "lens"]).best_layer.value_counts().unstack(fill_value=0)
    .reindex(columns=range(gpt.n_layers), fill_value=0)
)

## 5. How both lenses behave in general

Over all prompt positions (not only the readout one):

* **agreement** — the layer's top-1 equals the model's top-1: how deep the final prediction is formed;
* **copy rate** — the layer's top-1 is the input token at that position: the residual stream still "is" the token embedding
  (GPT-2 ties `wte` and `lm_head`, so an untouched embedding decodes to itself);
* **KL(model ‖ layer)** — how far the layer's distribution is from the output one.

In [ ]:
behaviour = {
    name: {column: layer_mean(items[items.lens == name], column)
           for column in ("agreement", "copy_rate", "kl_to_final")}
    for name in LENS_NAMES
}
plot_layer_curves(
    {f"{name}: top-1 == model top-1": behaviour[name]["agreement"] for name in LENS_NAMES}
    | {f"{name}: top-1 == input token": behaviour[name]["copy_rate"] for name in LENS_NAMES},
    title="Both lenses over all prompt positions",
    ylabel="share of positions",
)
plot_layer_curves(
    {f"{name}: KL(model || layer)": behaviour[name]["kl_to_final"] for name in LENS_NAMES},
    title="Both lenses: KL to the model output",
    ylabel="nats",
)

In [ ]:
behaviour_rows = []
for name in LENS_NAMES:
    agreement, copy_rate, kl = (behaviour[name][c] for c in ("agreement", "copy_rate", "kl_to_final"))
    for dataset in agreement.index:
        behaviour_rows.append({
            "dataset": dataset,
            "lens": name,
            "agreement ≥ 50% from": first_layer(agreement.loc[dataset], 0.5),
            "KL ≤ 1 nat from": first_layer(kl.loc[dataset], 1.0, above=False),
            "copy rate at L0": copy_rate.loc[dataset, 0],
            f"copy rate at L{LAST - 1}": copy_rate.loc[dataset, LAST - 1],
        })
pd.DataFrame(behaviour_rows).set_index(["dataset", "lens"]).unstack("lens").round(3)

## 6. What drives a hit

### 6a. The word is already in the prompt

Early layers of the logit lens decode the input token itself, so a word that is literally in the prompt
(e.g. a number in order-ops) can be a "hit" without any computation.

In [ ]:
hits = inter.assign(hit=inter.best_rank <= K, inner_hit=inter.inner_best <= K)
hits.groupby(["dataset", "in_prompt", "lens"]).hit.agg(["mean", "size"]).unstack(["in_prompt", "lens"]).round(3)

### 6b. Single-token words vs words that only have a first-token fallback

In [ ]:
hits.groupby(["dataset", "single_token", "lens"]).hit.agg(["mean", "size"]).unstack(["single_token", "lens"]).round(3)

### 6c. Role of the intermediate

multilingual: `[language, relation, source word, answer in English]`; order-ops: `[intermediate number, operation]`.

In [ ]:
roles = hits[hits.dataset.isin(list(ROLE_NAMES))].copy()
roles["role_name"] = [ROLE_NAMES[d][r] for d, r in zip(roles.dataset, roles.role)]
roles.groupby(["dataset", "role_name", "lens"]).agg(
    hit_rate=("hit", "mean"), median_best_rank=("best_rank", "median"), n=("hit", "size"),
    inner_hit_rate=("inner_hit", "mean"), median_inner_rank=("inner_best", "median"),
).unstack("lens").round(3)

In [ ]:
fig, axes = plt.subplots(1, len(ROLE_NAMES), figsize=(15, 4), sharey=True, squeeze=False)
for ax, dataset in zip(axes.flat, ROLE_NAMES):
    group = roles[roles.dataset == dataset]
    for (role, name), role_group in group.groupby(["role_name", "lens"]):
        ax.plot(
            (np.stack(role_group.ranks) <= K).mean(0), marker="o", ms=3,
            label=f"{role}: {name}", linestyle="--" if name == "logit lens" else "-",
        )
    ax.set_title(dataset)
    ax.set_xlabel("layer (last = model output)")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=7)
axes.flat[0].set_ylabel(f"rank ≤ {K}")
plt.tight_layout()
plt.show()

### 6d. Does the intermediate show up more often when the model gets the answer right?

In [ ]:
correctness = hits.merge(
    with_target[["dataset", "item", "model_correct"]], on=["dataset", "item"], validate="many_to_one"
)
correctness.groupby(["dataset", "model_correct", "lens"]).hit.agg(["mean", "size"]).unstack(["model_correct", "lens"]).round(3)

## 7. Items: successes and failures

In [ ]:
columns = ["dataset", "item", "lens", "word", "best_rank", "best_layer", "inner_best", "inner_best_layer", "in_prompt"]
inter.sort_values("best_rank").groupby(["dataset", "lens"]).head(3)[columns]

In [ ]:
inter.sort_values("best_rank", ascending=False).groupby(["dataset", "lens"]).head(3)[columns]

What the lens reads at the readout position, layer by layer (first items of each dataset):

In [ ]:
readout = items.groupby(["dataset", "lens"]).head(2).set_index(["dataset", "item", "lens"]).readout_top1.apply(pd.Series)
readout.columns.name = "layer"
readout.sort_index()[shown_layers(gpt.n_layers, LAYER_STRIDE)]

The most common readout top-1 tokens per layer over all items — generic tokens mean the lens shows no item-specific content there:

In [ ]:
top1_tables = {}
for name in LENS_NAMES:
    top1_by_layer = pd.DataFrame(items[items.lens == name].readout_top1.tolist())
    top1_tables[name] = pd.DataFrame({
        "distinct top-1 / items": top1_by_layer.nunique() / len(top1_by_layer),
        "most common": {
            layer: ", ".join(f"{token!r}×{count}" for token, count in column.value_counts().head(5).items())
            for layer, column in top1_by_layer.items()
        },
    }).rename_axis("layer")
pd.concat(top1_tables, axis=1, names=["lens", "metric"])

### 7a. Paired inner-layer ranks: which lens improves each intermediate?

One row per intermediate occurrence, aligned by dataset, item, word, and role. A positive
`log10 gain` means J-lens lowers the best inner-layer rank; ties are counted explicitly.

In [ ]:
head_to_head = inter.pivot(
    index=["dataset", "item", "word", "role"], columns="lens", values="inner_best"
)
head_to_head["winner"] = np.select(
    [head_to_head["J-lens"] < head_to_head["logit lens"],
     head_to_head["J-lens"] > head_to_head["logit lens"]],
    ["J-lens", "logit lens"], default="tie",
)
head_to_head["log10 gain"] = np.log10(head_to_head["logit lens"]) - np.log10(head_to_head["J-lens"])
paired_summary = head_to_head.groupby("dataset").winner.value_counts().unstack(fill_value=0)
paired_summary = paired_summary.reindex(columns=["J-lens", "logit lens", "tie"], fill_value=0)
paired_summary["median log10 gain"] = head_to_head.groupby("dataset")["log10 gain"].median()
display(paired_summary.round(3))
display(head_to_head.sort_values("log10 gain", ascending=False).head(15))
display(head_to_head.sort_values("log10 gain").head(15))

## 8. Conclusions

The cell below turns the numbers above into statements, so they stay correct after a re-run
(other seed, other `K`, other model size).

In [ ]:
def pct(x):
    return "n/a" if x is None or pd.isna(x) else f"{100 * x:.0f}%"


def layer(x):
    return "never" if x is None else f"L{x}"


def hit_rates(frame, column):
    """Hit rate (best rank <= K) split by a boolean column: {value: (rate, n)}."""
    hit = (frame.best_rank <= K).groupby(frame[column])
    return {value: (rate, n) for value, rate, n in zip(hit.mean().index, hit.mean(), hit.size())}

In [ ]:
lines = ["Model accuracy (top-1 == target): " + ", ".join(f"{d} {pct(a)}" for d, a in accuracy.items()) + "."]
for name in LENS_NAMES:
    lines.append(f"--- {name} ---")
    lens_at_k = at_k.xs(name, axis=1, level="lens")
    inner_lens_at_k = inner_at_k.xs(name, axis=1, level="lens")
    output_lens_at_k = output_at_k.xs(name, axis=1, level="lens")
    lens_inter = inter[inter.lens == name]
    inter_curve = layer_hit_rate(lens_inter, K)
    target_curve = layer_hit_rate(target[target.lens == name], K)
    agreement, copy_rate, kl = (behaviour[name][c] for c in ("agreement", "copy_rate", "kl_to_final"))
    lens_roles = roles[roles.lens == name]
    lens_correctness = correctness[correctness.lens == name]
    for dataset in lens_at_k.index:
        i, c = lens_at_k.loc[dataset, "intermediate"], lens_at_k.loc[dataset].get("control")
        verdict = "clearly above" if i >= c + 0.1 else "slightly above" if i > c else "NOT above"
        peak = inter_curve.loc[dataset].iloc[:-1]
        lines.append(
            f"{dataset}: intermediates pass@{K} {pct(i)} vs control {pct(c)} — {verdict} chance; "
            f"inner layers alone {pct(inner_lens_at_k.loc[dataset, 'intermediate'])}, model output alone "
            f"{pct(output_lens_at_k.loc[dataset, 'intermediate'])}; best inner layer L{peak.idxmax()} "
            f"({pct(peak.max())} of words at rank ≤ {K})."
        )

    for dataset in target_curve.index:
        curve = target_curve.loc[dataset]
        if curve.iloc[-1] == 0:
            lines.append(f"{dataset}: the target is never in the output top-{K}.")
        else:
            lines.append(
                f"{dataset}: the target is in the output top-{K} for {pct(curve.iloc[-1])} of items "
                f"and reaches half of that by {layer(first_layer(curve, curve.iloc[-1] / 2))}."
            )

    for dataset in agreement.index:
        lines.append(
            f"{dataset}: lens top-1 == model top-1 on ≥50% of positions from {layer(first_layer(agreement.loc[dataset], 0.5))}, "
            f"KL ≤ 1 nat from {layer(first_layer(kl.loc[dataset], 1.0, above=False))}; copy rate "
            f"{pct(copy_rate.loc[dataset, 0])} at L0 → {pct(copy_rate.loc[dataset, LAST - 1])} at L{LAST - 1}."
        )

    for column, yes, no in [
        ("in_prompt", "already in the prompt", "absent from it"),
        ("single_token", "single-token", "first-token fallback"),
    ]:
        rates = hit_rates(lens_inter, column)
        (r_yes, n_yes), (r_no, n_no) = rates.get(True, (None, 0)), rates.get(False, (None, 0))
        lines.append(f"Intermediates {yes}: hit rate {pct(r_yes)} (n={n_yes}) vs {pct(r_no)} (n={n_no}) {no}.")

    for dataset, group in lens_roles.groupby("dataset"):
        by_role = group.groupby("role_name").hit.mean().sort_values(ascending=False)
        lines.append(f"{dataset} by role: " + ", ".join(f"{name} {pct(rate)}" for name, rate in by_role.items()) + ".")

    by_correct = lens_correctness.groupby("model_correct").hit.mean()
    lines.append(
        f"Intermediate hit rate when the model is right: {pct(by_correct.get(True))}, when wrong: {pct(by_correct.get(False))}."
    )


for dataset in inner_at_k.index:
    row = inner_at_k.loc[dataset]
    j, l = row[("intermediate", "J-lens")], row[("intermediate", "logit lens")]
    jc, lc = row[("control", "J-lens")], row[("control", "logit lens")]
    lines.append(
        f"{dataset}: inner-layer pass@{K}, J-lens {pct(j)} vs logit lens {pct(l)} "
        f"(Δ {100 * (j - l):+.1f} pp); margins over control "
        f"{100 * (j - jc):+.1f} pp vs {100 * (l - lc):+.1f} pp."
    )
for dataset, row in paired_summary.iterrows():
    lines.append(
        f"{dataset}: J-lens ranks intermediates higher in {row.get('J-lens', 0):.0f} cases, "
        f"logit lens in {row.get('logit lens', 0):.0f}, ties {row.get('tie', 0):.0f}; "
        f"median log10 rank gain {row['median log10 gain']:+.2f}."
    )
for line in lines:
    print("•", line)

### How to read these numbers

* **Intermediates vs control.** Compare each lens with its control baseline and compare the
  margins between lenses. A larger raw hit rate alone does not establish better intermediate readout.
* **Inner layers vs model output.** Both lenses use the exact same model output. All-layer
  metrics preserve the original analysis; inner-layer pass@k and the paired ranks isolate
  differences in the lens. An inner-layer hit is suggestive, not proof of a reasoning step.
* **Copy rate.** High early-layer copy rates can explain hits for words already in the prompt.
  Compare both lenses' copy curves and the in-prompt split (§6a).
* **Agreement / KL.** Earlier agreement and lower KL describe fidelity to the model output;
  they do not by themselves show that intermediates are exposed better.
* **Model correctness.** Accuracy is measured once per item. The right/wrong split keeps
  both lens measurements attached to that same model answer.
* **Tokenization.** Multi-token words fall back to their first token, which can be generic.
  The single-token split (§6b) retains this diagnostic. Controls share the dataset topic.
* **Fit corpus.** A J-lens depends on its fitting corpus and selected model. Use an independent
  corpus, keep the checkpoint with its model, and vary the mix to test sensitivity.